# AQUA-SENTINEL — GPU Training Pipeline (SIH26057)

**Runtime requirement: GPU.** `Runtime → Change runtime type → T4 GPU` before running.
Cell 1 hard-fails without a GPU by design.

Pipeline: upload `aqua_train_bundle.zip` (built locally via `make train-bundle`) →
prepare site-level splits → train YOLOv8 + U-Net + classifier → evaluate honestly →
export ONNX → download `aqua_onnx_models.zip` → drop into `models/weights/`.

Every metric printed is computed by the code — nothing is hard-coded.

In [ ]:
# @title 1 ▶ Environment check — FAILS LOUDLY WITHOUT GPU
import sys

import torch

if not torch.cuda.is_available():
    print("=" * 70)
    print("NO GPU DETECTED — TRAINING IS BLOCKED BY DESIGN.")
    print("Fix: Runtime → Change runtime type → Hardware accelerator → GPU (T4)")
    print("Then: Runtime → Restart session, and re-run from this cell.")
    print("=" * 70)
    sys.exit(3)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

In [ ]:
# @title 2 ▶ Upload `aqua_train_bundle.zip` (from `make train-bundle`)
import pathlib

if not pathlib.Path("/content/aqua_train_bundle.zip").exists():
    from google.colab import files

    print("Choose aqua_train_bundle.zip (~2 GB)…")
    up = files.upload()
    name = next(iter(up))
    if name != "aqua_train_bundle.zip":
        import os

        os.rename(name, "aqua_train_bundle.zip")
!unzip -q -o aqua_train_bundle.zip -d /content/aqua_train
!ls /content/aqua_train/bundle

In [ ]:
# @title 3 ▶ Install pinned deps
%pip install -q -r training/requirements-colab.txt

In [ ]:
# @title 4 ▶ Stage 2 — build leak-safe site-level splits
%cd /content/aqua_train
!python training/prepare_detection.py
!python training/prepare_segmentation.py --emit-filelists
!python training/prepare_classifier.py

In [ ]:
# @title 5 ▶ Train YOLOv8 detection (per-dataset models)
%cd /content/aqua_train
!python training/train_detection.py

In [ ]:
# @title 6 ▶ Train U-Net segmentation (wreck masks)
%cd /content/aqua_train
!python training/train_segmentation.py

In [ ]:
# @title 7 ▶ Train natural-vs-artificial classifier (64×64 crops)
%cd /content/aqua_train
!python training/train_classifier.py

In [ ]:
# @title 8 ▶ Export ONNX (app-compatible) + numeric verification
%cd /content/aqua_train
!python training/export_onnx.py

In [ ]:
# @title 9 ▶ Independent evaluation of exported ONNX (trust-but-verify gate)
%cd /content/aqua_train
!python training/evaluate_onnx.py

In [ ]:
# @title 10 ▶ Download models + all metrics
from google.colab import files

files.download("/content/aqua_train/runs/aqua_onnx_models.zip")
files.download("/content/aqua_train/runs/onnx_evaluation.json")
print("Next steps (local):")
print("  1. unzip into models/weights/")
print("  2. models/registry.json: backend=onnx, path=models/weights/<file>")
print("  3. restart uvicorn — pipeline now runs the trained models;")
print("     missing weights still fall back to heuristics automatically.")